# Шаг 5. Запуск сервиса из Jupyter / Colab

Основной способ запуска — Docker Compose (`docker compose up -d`, Swagger: http://localhost:8000/docs).
Этот ноутбук показывает тот же сервис внутри Jupyter: uvicorn в фоновом потоке, вариант из курса с `nest_asyncio` и проверку через `TestClient`.

Локально ставьте зависимости только в `.venv` проекта, в глобальный Python — нет. Ноутбук открывайте из папки `inference-api/notebooks/`.

In [ ]:
# В Colab: склонировать репозиторий и поставить зависимости (локально — уже в .venv)
# !git clone https://github.com/pudoff/SpamClickbaitClassification.git && %cd SpamClickbaitClassification/inference-api/notebooks
# %pip install -q -r ../requirements-dev.txt

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent  # inference-api/
sys.path.insert(0, str(ROOT))
os.environ.setdefault("MODEL_PATH", str(ROOT / "models" / "model.joblib"))
print("MODEL_PATH =", os.environ["MODEL_PATH"], "| exists:", Path(os.environ["MODEL_PATH"]).exists())

In [ ]:
import threading
import time

import uvicorn

from app.main import create_app

# uvicorn в отдельном потоке со своим event loop: ячейка не блокируется и nest_asyncio не нужен.
app = create_app()
server = uvicorn.Server(uvicorn.Config(app, host="127.0.0.1", port=8001, log_level="info"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)
print("Swagger UI: http://127.0.0.1:8001/docs")

### Вариант из курса: `nest_asyncio`

В Jupyter/Colab уже работает event loop, поэтому `uvicorn.run(app)` в ячейке падает с ошибкой «asyncio.run() cannot be called from a running event loop». `nest_asyncio.apply()` разрешает вложенный цикл, но ячейка **блокируется**, пока сервер не остановят (■ Interrupt).

⚠️ На Python 3.12+ (сейчас и в Colab) `nest_asyncio` ломает anyio: запросы падают с 500 (`cannot create weak reference to 'NoneType'`). Поэтому ячейка ниже закомментирована, а основной способ — поток выше. Не запускайте оба варианта в одном ядре: `apply()` патчит asyncio глобально.

In [ ]:
# import nest_asyncio
# nest_asyncio.apply()
# uvicorn.run(create_app(), host="127.0.0.1", port=8002)  # блокирует ячейку

In [ ]:
import requests

BASE = "http://127.0.0.1:8001"
print(requests.get(f"{BASE}/healthcheck").json())
print(requests.get(f"{BASE}/model-info").json())
print(requests.post(f"{BASE}/predict", json={"text": "Вам одобрен кредит! Перезвоните по номеру"}).json())
# Валидация (шаг 6): порог вне [0, 1] -> 422
print(requests.post(f"{BASE}/predict", json={"text": "привет", "threshold": 1.5}).status_code)

In [ ]:
from fastapi.testclient import TestClient

# TestClient вызывает приложение напрямую, без сети и без запущенного uvicorn
with TestClient(create_app()) as client:
    for text in [
        "Скидка 90% только сегодня, переходите по ссылке!",
        "Вы не поверите, что сделала эта звезда! Шок-фото",
        "Буду через десять минут, возьми хлеба",
    ]:
        r = client.post("/predict", json={"text": text})
        print(r.status_code, r.json()["status"], r.json()["label"], round(r.json()["confidence"], 3), "|", text)

In [ ]:
server.should_exit = True  # остановить фоновый uvicorn